In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (O MÉTODO DAS OPÇÕES LOCAIS)
# ============================================================
import os
from dotenv import load_dotenv
from pyspark.sql.functions import current_timestamp, input_file_name, year, month, col, to_timestamp, count, avg, round, when, regexp_replace, coalesce, lit
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient


# Carrega o .env
load_dotenv("../env")

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# Empacotando as credenciais em um dicionário de opções
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Cria a credencial usando as variáveis
credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

# Conecta no serviço do Data Lake
service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

tabela = "ecommerce_produtos"
path_raw = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data/{tabela}.csv"
path_bronze = f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/{tabela}"

print("Credenciais empacotadas e caminhos configurados.")

In [0]:
# ============================================================
# 2. LEITURA E AUDITORIA (RAW -> BRONZE)
# ============================================================


print(f"Lendo {tabela} com opções de autenticação localizadas...")

# Usando o .options(**adls_options) para abrir a porta do Data Lake apenas para esta leitura
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .options(**adls_options) 
    .load(path_raw)
)

print(f"Leitura concluída! Total de registros: {df_raw.count()}")

In [0]:
# ============================================================
# 3. VALIDAÇÃO DO CONTRATO DE DADOS (SCHEMA ENFORCEMENT)
# ============================================================
colunas_esperadas = [
    "sku", "nome_produto", "descricao", "id_categoria", 
    "preco_lista", "unidade_medida", "nome_marca", "is_ativo"
]

def validar_contrato_colunas(df, colunas_esperadas):
    colunas_atuais = df.columns
    if len(colunas_atuais) != len(colunas_esperadas):
        raise ValueError(f"ERRO CRÍTICO: O arquivo possui {len(colunas_atuais)} colunas, esperado {len(colunas_esperadas)}.")
    
    colunas_faltantes = set(colunas_esperadas) - set(colunas_atuais)
    colunas_extras = set(colunas_atuais) - set(colunas_esperadas)
    
    if colunas_faltantes or colunas_extras:
        raise ValueError(f"ERRO CRÍTICO: Faltam {list(colunas_faltantes)} / Sobram {list(colunas_extras)}")
        
    if colunas_atuais != colunas_esperadas:
        raise ValueError("ERRO CRÍTICO: A ordem das colunas está incorreta.")
    
    print("Validação estrutural OK: quantidade, nomes e ordem das colunas corretos.")

# Executa a validação
validar_contrato_colunas(df_raw, colunas_esperadas)

In [0]:
# Usa a data atual da carga para particionar (já que produtos não tem data na tabela)

timestamp_carga = current_timestamp()

df_raw = (
    df_raw
    .withColumn("bronze_ingested_at", timestamp_carga)
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("ano_particao", year(timestamp_carga))
    .withColumn("mes_particao", month(timestamp_carga))
)

display(df_raw.limit(5))

In [0]:

# ============================================================
# 4. CARGA FÍSICA (WRITE BRONZE)
# ============================================================
print(f"Iniciando gravação física na camada Bronze...")

(
    df_raw.write
    .format("delta")
    .mode("append") 
    .options(**adls_options) # Repassando as credenciais para a gravação garantir o acesso
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print(f"SUCESSO! Ingestão da camada Raw para a Bronze finalizada.")

In [0]:
# ============================================================
# 5. VALIDAÇÃO DA GRAVAÇÃO (LENDO A TABELA DELTA)
# ============================================================
print(f"Buscando os dados gravados na camada Bronze...\nCaminho: {path_bronze}\n")

# Lendo o formato "delta" passando as credenciais que já estão na memória
df_bronze_salvo = (
    spark.read
    .format("delta")
    .options(**adls_options) 
    .load(path_bronze)
)

# Mostra a volumetria real que está no Data Lake
print(f"Total de registros físicos na tabela Delta: {df_bronze_salvo.count()}")

# Exibe os dados, ordenando pelos mais recentes para você ver as colunas de auditoria e partição
display(
    df_bronze_salvo
    .orderBy(col("bronze_ingested_at").desc())
)

In [0]:

# ============================================================
# 6. CAMADA SILVER (APLICAÇÃO DAS REGRAS TÉCNICAS)
# ============================================================

print(f"Iniciando processamento da camada Silver para: {tabela}...")

# Define o caminho de destino na Silver
path_silver = f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/{tabela}"

# 1. LEITURA: Lendo a tabela Delta da Bronze
df_bronze_read = (
    spark.read
    .format("delta")
    .options(**adls_options) 
    .load(path_bronze)
)

# 2. TRANSFORMAÇÃO E QUALIDADE (APLICAÇÃO DO CONTRATO)
df_silver = (
    df_bronze_read
    
    # ==========================================================
    # REGRA 1: sku não nulo e único na Silver
    # ==========================================================
    .dropna(subset=["sku"]) # Remove as linhas onde o SKU é vazio/nulo
    .dropDuplicates(["sku"]) # Garante a unicidade da Primary Key
    
    # Tipagens básicas de texto
    .withColumn("sku", col("sku").cast("string"))
    .withColumn("nome_produto", col("nome_produto").cast("string"))
    .withColumn("descricao", col("descricao").cast("string"))
    .withColumn("unidade_medida", col("unidade_medida").cast("string"))
    .withColumn("nome_marca", col("nome_marca").cast("string"))
    .withColumn("id_categoria", col("id_categoria").cast("string"))
    
    # ==========================================================
    # REGRA 2: preco_lista castado como DECIMAL(10,2) e > 0
    # ==========================================================
    # 1º Trocando a vírgula por ponto (caso exista)
    # 2º Convertendo rigorosamente para formato financeiro Decimal(10,2)
    .withColumn("preco_lista", regexp_replace(col("preco_lista"), ",", ".").cast("decimal(10,2)"))
    # 3º Filtrando para manter apenas produtos com preço válido (maior que zero)
    .filter(col("preco_lista") > 0)
    
    # ==========================================================
    # REGRA 3: is_ativo castado como booleano; nulos imputados como False
    # ==========================================================
    # O coalesce olha para o valor: se for nulo, ele substitui pelo que está no lit(False)
    .withColumn("is_ativo", coalesce(col("is_ativo").cast("boolean"), lit(False)))
    
    # Auditoria da camada Silver
    .withColumn("silver_processed_at", current_timestamp())
)

# 3. CARGA: Gravando na Silver
print(f"Gravando dados limpos fisicamente em: {path_silver}")

(
    df_silver.write
    .format("delta")
    .mode("append") 
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver)
)

print("SUCESSO! Regras Técnicas aplicadas e dados gravados na camada Silver.\n")

print("=== NOVO SCHEMA DA TABELA ===")
df_silver.printSchema()

display(df_silver.limit(5))

In [0]:
# COMMAND ----------
# ============================================================
# 7. CAMADA GOLD (PRODUTOS - AGREGAÇÃO SEMESTRAL E CONTROLE)
# ============================================================
from pyspark.sql.functions import col, avg, round, current_timestamp, year, month, when, date_format

print("A iniciar o processamento da camada Gold (Média de Preço por Semestre)...")

# Caminho de destino do Data Mart na Gold
path_gold_agg_categorias = f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/hist_preco_medio_semestre"

# A ler a tabela Delta da Silver 
df_silver = (
    spark.read
    .format("delta")
    .options(**adls_options) 
    .load(path_silver)
)

# ------------------------------------------------------------
# TRANSFORMAÇÃO E AGREGAÇÃO SEMESTRAL
# ------------------------------------------------------------
df_gold_agg = (
    df_silver
    .filter(col("is_ativo") == True) 
    
    .withColumn("data_fotografia", current_timestamp())
    .withColumn("ano_particao", year(col("data_fotografia")))
    .withColumn("mes", month(col("data_fotografia")))
    .withColumn("semestre_particao", when(col("mes") <= 6, 1).otherwise(2))
    
    .groupBy("id_categoria", "ano_particao", "semestre_particao")
    .agg(
        round(avg("preco_lista"), 2).alias("preco_medio_semestre")
    )
    
    # Máscara de formatação relacional (YYYY-MM-DD HH24:MI:SS)
    .withColumn("gold_processed_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

# ============================================================
# 8. CARGA FÍSICA (WRITE GOLD - CHUMBADO EM APPEND)
# ============================================================
print("A gravar o Data Mart na camada Gold (MODO: APPEND - HISTÓRICO ATIVADO)...")

(
    df_gold_agg.write
    .format("delta")
    .mode("append") # <--- Comportamento definitivo de produção
    .option("mergeSchema", "true") # <--- Essencial para a segurança do schema no append
    .options(**adls_options)
    .partitionBy("ano_particao", "semestre_particao")
    .save(path_gold_agg_categorias)
)

print("SUCESSO! Agregação semestral acumulada na Gold via APPEND.\n")
display(df_gold_agg.limit(5))

In [0]:
# ============================================================
# 9. EXPORTAÇÃO PARA O SQL SERVER (SERVING LAYER)
# ============================================================

print("Iniciando a exportação do Data Mart para o SQL Server...")

# Garante que as variáveis do .env estão carregadas
load_dotenv(".env")

# 1. Configurações de Conexão (Lendo de forma segura do .env)
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port = "1433" 
jdbc_database = os.getenv("SQL_DATABASE")

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

# Nome da tabela que será criada/atualizada no banco de dados
tabela_destino = "dbo.PrecoMedioSemestre" 

# 2. Gravando no SQL Server via Conector Nativo (Padrão Serverless)
try:
    (
        df_gold_agg.write
        .format("sqlserver")
        # Substituímos a "url" pelas opções segregadas que o Serverless exige:
        .option("host", jdbc_hostname)
        .option("port", jdbc_port)
        .option("database", jdbc_database)
        .option("dbtable", tabela_destino)
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .mode("overwrite") # Usando overwrite na primeira carga
        .save()
    )
    print(f"SUCESSO! Dados exportados perfeitamente para a tabela {tabela_destino} no SQL Server.")
except Exception as e:
    print(f"Erro ao exportar para o SQL Server:\n{e}")